# Latent MNAR DGP pilot

## tl;dr

Across 50 replications with $n=500$, the Allman-inspired binary DGP gives the cleanest theorem-aligned primary design: the oracle bridge has $M_j$ RMSE 0.0565 and $p_f$ RMSE 0.0642, versus 0.0403 and 0.0444 under full data. The Zhao-style normal hybrid is a useful secondary design; the Poisson hybrid is better treated as a stress test because bridge weighting roughly doubles its measurement-kernel RMSE. These are pilot results, not the final 1,000-replication study.

## Context & Methods

Zhao and Shao (2015) do not include a latent class, so their DGP cannot directly evaluate $M_j$ or $p_f$. The `zhao_hybrid_*` profiles retain their outcome-family and MNAR-logit logic while adding a shared binary latent class and three conditionally independent items. The `allman_binary` profile is a finite categorical three-view model aligned with the current identification theorem.

### Key Assumptions

- $F\in\{0,1\}$ and $W$ shifts $P(F=1\mid W)$.
- Three items are conditionally independent given $F$.
- Each missingness indicator depends on its own item and $U=(X,W)$, but not directly on $Z$.
- Mean item observation is calibrated to 80%; pair and complete-case rates are checked separately.
- The pilot oracle bridge uses true pair propensities. Estimated bridges are a later stage.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'simulation':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from simulation.latent_mnar_sim import load_config, run_pilot

CONFIG_PATH = ROOT / 'simulation' / 'config.yml'
config = load_config(CONFIG_PATH)
print(f"n={config['sample_size']}, pilot replications={config['pilot_replications']}")

n=500, pilot replications=50


## Data

Run every candidate DGP with the same sample size, latent shift, target observation rate, supported pairs, and fitting settings.

In [2]:
summary, diagnostics = run_pilot(config)
results_dir = ROOT / 'simulation' / 'results'
results_dir.mkdir(exist_ok=True)
summary.to_csv(results_dir / 'dgp_pilot_summary.csv', index=False)
diagnostics.to_csv(results_dir / 'dgp_pilot_diagnostics.csv', index=False)
summary.round(4)

,profile,family,method,M_bias,M_mean_absolute_error,p_f1_bias,mean_iterations,M_RMSE,M_normalized_RMSE,p_f1_RMSE
0,allman_binary,binary,full_data_oracle,0.0047,0.0328,-0.0016,47.26,0.0403,0.0671,0.0444
1,allman_binary,binary,mar_naive,-0.0405,0.0539,-0.0713,60.06,0.0686,0.1143,0.0923
2,allman_binary,binary,oracle_bridge,0.0034,0.0448,-0.0013,62.96,0.0565,0.0941,0.0642
3,zhao_hybrid_binary,binary,full_data_oracle,-0.0052,0.0397,0.0127,67.04,0.0505,0.1010,0.0672
4,zhao_hybrid_binary,binary,mar_naive,-0.0615,0.0719,-0.0422,82.72,0.0886,0.1773,0.0897
5,zhao_hybrid_binary,binary,oracle_bridge,-0.0096,0.0532,0.0150,86.94,0.0669,0.1339,0.0897
6,zhao_hybrid_normal,normal,full_data_oracle,0.0083,0.0516,-0.0060,10.18,0.0641,0.0308,0.0286
7,zhao_hybrid_normal,normal,mar_naive,-0.0919,0.1063,-0.0929,10.84,0.1318,0.0632,0.0990
8,zhao_hybrid_normal,normal,oracle_bridge,0.0149,0.0741,-0.0111,10.04,0.0932,0.0447,0.0356
9,zhao_hybrid_poisson,poisson,full_data_oracle,-0.0011,0.0802,-0.0035,27.22,0.1042,0.0601,0.0337


## Results

### Missingness and overlap checks

In [3]:
diagnostic_summary = (
    diagnostics.groupby('profile')[
        ['item_observation_rate', 'anchor_pair_rate', 'extension_pair_rate',
         'complete_case_rate', 'minimum_item_propensity']
    ].mean()
)
diagnostic_summary.round(4)

,item_observation_rate,anchor_pair_rate,extension_pair_rate,complete_case_rate,minimum_item_propensity
profile,,,,,
allman_binary,0.7975,0.6460,0.6394,0.5206,0.4950
zhao_hybrid_binary,0.8014,0.6462,0.6461,0.5249,0.4800
zhao_hybrid_normal,0.7978,0.6454,0.6398,0.5247,0.2880
zhao_hybrid_poisson,0.8025,0.6525,0.6516,0.5326,0.1102


### Primary latent-parameter metrics

Smaller absolute bias and RMSE are better. Signed $M$ bias is retained as a directional diagnostic, while `M_mean_absolute_error` prevents cancellation across items and classes. `M_normalized_RMSE` divides by the mean class separation so that outcome families with different units can be compared cautiously.

In [4]:
primary_columns = [
    'profile', 'family', 'method', 'M_bias', 'M_mean_absolute_error',
    'M_RMSE', 'M_normalized_RMSE', 'p_f1_bias', 'p_f1_RMSE'
]
summary[primary_columns].sort_values(['profile', 'method']).round(4)

,profile,family,method,M_bias,M_mean_absolute_error,M_RMSE,M_normalized_RMSE,p_f1_bias,p_f1_RMSE
0,allman_binary,binary,full_data_oracle,0.0047,0.0328,0.0403,0.0671,-0.0016,0.0444
1,allman_binary,binary,mar_naive,-0.0405,0.0539,0.0686,0.1143,-0.0713,0.0923
2,allman_binary,binary,oracle_bridge,0.0034,0.0448,0.0565,0.0941,-0.0013,0.0642
3,zhao_hybrid_binary,binary,full_data_oracle,-0.0052,0.0397,0.0505,0.1010,0.0127,0.0672
4,zhao_hybrid_binary,binary,mar_naive,-0.0615,0.0719,0.0886,0.1773,-0.0422,0.0897
5,zhao_hybrid_binary,binary,oracle_bridge,-0.0096,0.0532,0.0669,0.1339,0.0150,0.0897
6,zhao_hybrid_normal,normal,full_data_oracle,0.0083,0.0516,0.0641,0.0308,-0.0060,0.0286
7,zhao_hybrid_normal,normal,mar_naive,-0.0919,0.1063,0.1318,0.0632,-0.0929,0.0990
8,zhao_hybrid_normal,normal,oracle_bridge,0.0149,0.0741,0.0932,0.0447,-0.0111,0.0356
9,zhao_hybrid_poisson,poisson,full_data_oracle,-0.0011,0.0802,0.1042,0.0601,-0.0035,0.0337


## Takeaways

The pilot supports `allman_binary` as the primary DGP because it directly matches the finite-category theorem and both latent targets remain stable after oracle bridge weighting. `zhao_hybrid_normal` is the strongest secondary design for showing continuity with Zhao and Shao's outcome/MNAR setup. `zhao_hybrid_poisson` should initially be a stress test, not the headline design. MAR-naive bias is visible in all profiles, but DGP selection is based first on oracle recoverability and overlap, not on making MAR look poor.

In [5]:
metric_view = summary[
    ['profile', 'method', 'M_RMSE', 'M_normalized_RMSE', 'p_f1_RMSE']
].copy()
full = metric_view[metric_view['method'] == 'full_data_oracle'].set_index('profile')
bridge = metric_view[metric_view['method'] == 'oracle_bridge'].set_index('profile')
comparison = bridge[['M_RMSE', 'M_normalized_RMSE', 'p_f1_RMSE']].copy()
comparison['M_RMSE_inflation_vs_full'] = comparison['M_RMSE'] / full['M_RMSE']
comparison['p_f1_RMSE_inflation_vs_full'] = comparison['p_f1_RMSE'] / full['p_f1_RMSE']
comparison.sort_values('M_normalized_RMSE').round(4)

,M_RMSE,M_normalized_RMSE,p_f1_RMSE,M_RMSE_inflation_vs_full,p_f1_RMSE_inflation_vs_full
profile,,,,,
zhao_hybrid_normal,0.0932,0.0447,0.0356,1.4545,1.2445
allman_binary,0.0565,0.0941,0.0642,1.4023,1.4458
zhao_hybrid_poisson,0.2135,0.1231,0.0572,2.0487,1.7010
zhao_hybrid_binary,0.0669,0.1339,0.0897,1.3260,1.3349
